# Project: counting cells in microscopy images

Threshold, label and measure bright objects in images with numpy and `scipy.ndimage`, for a whole folder of images.

The full explanations are in [counting_cells.md](counting_cells.md). *This work is licensed under [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).*

## How to use this notebook

* Run the cells **in order**, with `Shift+Enter`.
* Cells starting with `%%writefile` **create a file** (module, tests or script)
  in the folder of this notebook: like in the lesson, the project is made of
  real `.py` files, that you can also open in a text editor or Spyder.
* `%run script.py` runs a script as from the shell, and the cell with
  `pytest` runs the tests.
* Start JupyterLab from an **empty folder** dedicated to this project, in the
  `sci` environment:

  ```
  (sci) $ mkdir my_project
  (sci) $ cd my_project
  (sci) $ jupyter lab
  ```

## Step 1: the module

`make_image` creates synthetic images whose content we know; `threshold`
separates objects from the background with robust statistics (median and
MAD); `measure` labels and measures the objects.

In [ ]:
%%writefile cells.py
"""Detect and measure bright objects (e.g. cell nuclei) in an image."""

import numpy as np
from scipy import ndimage


def make_image(rng, n_cells=30, size=256, noise=0.15):
    """Create a synthetic fluorescence image with bright disks on a dark background.

    Returns
    -------
    image : numpy.ndarray
        2D float array of shape (size, size).
    """
    y, x = np.mgrid[0:size, 0:size]
    image = np.zeros((size, size))
    for _ in range(n_cells):
        cy, cx = rng.integers(10, size - 10, 2)
        radius = rng.uniform(4, 9)
        brightness = rng.uniform(0.6, 1.0)
        image[(x - cx)**2 + (y - cy)**2 <= radius**2] = brightness
    image = ndimage.gaussian_filter(image, 1.5)      # blur, as a microscope
    image += rng.normal(0.1, noise, image.shape)       # camera noise
    return image


def threshold(image, k=3):
    """Return a mask of the pixels brighter than the background.

    The background level is estimated with the median and the noise with
    the median absolute deviation, which are not influenced much by the
    bright objects.
    """
    background = np.median(image)
    noise = np.median(np.abs(image - background)) * 1.4826
    return image > background + k * noise


def measure(mask, min_area=20):
    """Label the objects of a mask and measure them.

    Parameters
    ----------
    mask : numpy.ndarray of bool
        Pixels belonging to objects.
    min_area : int
        Objects smaller than this (in pixels) are ignored, as noise.

    Returns
    -------
    labels : numpy.ndarray of int
        Image where each kept object has its own number (0 = background).
    areas : numpy.ndarray
        Area of each kept object, in pixels.
    centers : list of (float, float)
        Center (row, column) of each kept object.
    """
    labels, n = ndimage.label(mask)
    areas = np.bincount(labels.ravel())[1:]          # pixels per object
    keep = np.flatnonzero(areas >= min_area) + 1      # labels of large objects
    labels = np.where(np.isin(labels, keep), labels, 0)
    centers = ndimage.center_of_mass(mask, labels, keep)
    return labels, areas[keep - 1], centers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from cells import make_image, threshold, measure

image = make_image(np.random.default_rng(0), n_cells=30)
print(image.shape, image.dtype)
plt.imshow(image, cmap="gray")
plt.colorbar()

## Step 2: thresholding

Try different values of `k`: too low, noise becomes objects; too high, faint cells disappear.

In [ ]:
mask = threshold(image, k=3)
print("Proportion of object pixels:", mask.mean())
plt.imshow(mask)

## Step 3: labelling and measuring

In [ ]:
labels, areas, centers = measure(mask)
print(len(areas), "objects")
plt.imshow(labels, cmap="nipy_spectral")

## Step 4: tests

On noise-free images, we know the right answer exactly.

In [ ]:
%%writefile test_cells.py
"""Tests for cells.py."""

import numpy as np

from cells import measure, threshold


def disk_image(centers, radius=6, size=100):
    """Image with bright disks of the given radius, no noise."""
    y, x = np.mgrid[0:size, 0:size]
    image = np.zeros((size, size))
    for cy, cx in centers:
        image[(x - cx)**2 + (y - cy)**2 <= radius**2] = 1.0
    return image


def test_counts_separated_disks():
    mask = disk_image([(20, 20), (50, 70), (80, 30)]) > 0.5
    labels, areas, centers = measure(mask)
    assert len(areas) == 3


def test_area_of_a_disk():
    mask = disk_image([(50, 50)], radius=10) > 0.5
    labels, areas, centers = measure(mask)
    assert abs(areas[0] - np.pi * 10**2) < 0.05 * np.pi * 10**2


def test_center_of_a_disk():
    mask = disk_image([(30, 60)]) > 0.5
    labels, areas, centers = measure(mask)
    assert np.allclose(centers[0], (30, 60))


def test_small_objects_are_ignored():
    mask = np.zeros((50, 50), dtype=bool)
    mask[10, 10] = True                     # one isolated pixel
    labels, areas, centers = measure(mask, min_area=5)
    assert len(areas) == 0


def test_noise_only_gives_few_pixels():
    rng = np.random.default_rng(0)
    image = rng.normal(0.1, 0.15, (200, 200))
    assert threshold(image).mean() < 0.01

In [ ]:
import sys
!{sys.executable} -m pytest -q

## Step 5: processing a folder of images

In [ ]:
%%writefile generate_images.py
"""Generate a few synthetic microscopy images."""

from pathlib import Path

import numpy as np

from cells import make_image

rng = np.random.default_rng(seed=3)
folder = Path("images")
folder.mkdir(exist_ok=True)
for i, n_cells in enumerate([15, 30, 45]):
    np.save(folder / f"field_{i + 1}.npy", make_image(rng, n_cells))
    print("Written", folder / f"field_{i + 1}.npy")

In [ ]:
%run generate_images.py

In [ ]:
%%writefile count_cells.py
"""Count the cells in all the images of a folder."""

import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from cells import measure, threshold


def save_overlay(image, centers, filename):
    """Save the image with a red cross on each detected object."""
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(image, cmap="gray")
    rows = [c[0] for c in centers]
    cols = [c[1] for c in centers]
    ax.plot(cols, rows, "r+", markersize=8)
    ax.set_title(f"{len(centers)} objects")
    ax.axis("off")
    fig.savefig(filename, dpi=100)
    plt.close(fig)


def main():
    parser = argparse.ArgumentParser(description="Count bright objects in images.")
    parser.add_argument("folder", help="folder containing .npy images")
    parser.add_argument("--k", type=float, default=3, help="threshold level")
    parser.add_argument("--min-area", type=int, default=20,
                        help="minimum object area (pixels)")
    args = parser.parse_args()

    for filename in sorted(Path(args.folder).glob("*.npy")):
        image = np.load(filename)
        mask = threshold(image, args.k)
        labels, areas, centers = measure(mask, args.min_area)
        print(f"{filename.stem}: {len(areas)} objects, "
              f"mean area {areas.mean():.0f} px")
        save_overlay(image, centers, filename.with_suffix(".png"))


if __name__ == "__main__":
    main()

In [ ]:
%run count_cells.py images

We created 15, 30 and 45 cells. **Look at the result images**: why are fewer cells found?

In [ ]:
from IPython.display import Image
Image("images/field_3.png")

Touching cells merge into one object, and faint cells may stay below the threshold: **always check an analysis visually**.

## Going further

**1.** Plot the histogram of the areas of the objects found in `field_3`. Which objects are probably merged cells?

<details><summary>Answer</summary>

```python
image = np.load("images/field_3.npy")
labels, areas, centers = measure(threshold(image))
plt.hist(areas, bins=20)
plt.xlabel("Area (pixels)")
```

Objects about twice larger than a single cell (around 100 pixels) are probably merged cells.

</details>

In [ ]:
# Your code here

**2.** Compute the mean intensity of each object with `ndimage.mean(image, labels, index)`, and plot it against the area.

<details><summary>Answer</summary>

```python
from scipy import ndimage

index = np.unique(labels)[1:]
intensities = ndimage.mean(image, labels, index)
plt.scatter(areas, intensities)
```

</details>

In [ ]:
# Your code here